In [ ]:
%pip install pandas numpy scikit-learn joblib openpyxl
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor


# -----------------------------
# Configuration
# -----------------------------

MODEL_PATH = "energy_model.pkl"
TRAIN_DATASET = "/Users/sainishanth/Desktop/ML/Test/training_data.xlsx"
NEW_BATCH_DATA = "/Users/sainishanth/Desktop/ML/Test/new_batches.xlsx"

ERROR_THRESHOLD = 0.15


# -----------------------------
# Load model
# -----------------------------

model = joblib.load(MODEL_PATH)


# -----------------------------
# Load new batch data
# -----------------------------

new_data = pd.read_excel(NEW_BATCH_DATA)


X_new = new_data.drop(columns=["Power_Consumption_kW"])
y_actual = new_data["Power_Consumption_kW"]


# -----------------------------
# Run predictions
# -----------------------------

# prepare features exactly as training data
X_new_for_pred = X_new.drop(columns=["Batch_ID", "Time_Minutes"])
X_new_for_pred = pd.get_dummies(X_new_for_pred, columns=["Phase"])
for col in model.feature_names_in_:
    if col not in X_new_for_pred.columns:
        X_new_for_pred[col] = 0
X_new_for_pred = X_new_for_pred[model.feature_names_in_]

predictions = model.predict(X_new_for_pred)


# -----------------------------
# Evaluate model performance
# -----------------------------

mae = mean_absolute_error(y_actual, predictions)
rmse = np.sqrt(mean_squared_error(y_actual, predictions))

print("Model Performance on New Data")
print("MAE:", mae)
print("RMSE:", rmse)


# -----------------------------
# Log prediction results
# -----------------------------

log_df = X_new.copy()
log_df["Actual_Energy"] = y_actual
log_df["Predicted_Energy"] = predictions

log_df.to_excel("prediction_log.xlsx", index=False)


# -----------------------------
# Drift detection
# -----------------------------

if mae > ERROR_THRESHOLD:
    print("Performance drift detected. Retraining model...")

    # Load original training dataset
    dataset = pd.read_excel(TRAIN_DATASET)

    # Append new batch data
    dataset = pd.concat([dataset, new_data], ignore_index=True)

    # prepare features exactly as training data
    X = dataset.drop(columns=["Power_Consumption_kW", "Batch_ID", "Time_Minutes"])
    X = pd.get_dummies(X, columns=["Phase"])
    for col in model.feature_names_in_:
        if col not in X.columns:
            X[col] = 0
    X = X[model.feature_names_in_]

    y = dataset["Power_Consumption_kW"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Retrain model
    new_model = RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )

    new_model.fit(X_train, y_train)

    # Save updated model
    joblib.dump(new_model, "energy_model_v2.pkl")

    print("New model trained and saved as energy_model_v2.pkl")

else:
    print("Model performance acceptable. No retraining required.")


Note: you may need to restart the kernel to use updated packages.


/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model Performance on New Data
MAE: 2.523120469382562
RMSE: 3.967995684108458
Performance drift detected. Retraining model...
New model trained and saved as energy_model_v2.pkl
